# SignalObjExamples

Python port of the MATLAB `SignalObjExamples` helpfile (`helpfiles/SignalObjExamples.m`). Faithfully reproduces each example figure using the `nstat.SignalObj` API.

In [ ]:
# nSTAT-python notebook example: SignalObjExamples
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

from nstat import SignalObj
from nstat.notebook_figures import FigureTracker

np.random.seed(0)
OUTPUT_ROOT = REPO_ROOT / "output" / "notebook_images"
# MATLAB publish() emits 20 PNGs for this helpfile: Figs 1-15 cover Examples
# 1-5, Figs 16-17 are the Example 6 "signals + plotAllVariability" and
# customized-variability-bands renderings, Figs 18-19 are the per-label
# plotVariability(v1) / plotVariability(v2) figures, and Fig 20 is a side-by-
# side combined plotVariability(v1, v2) summary. We reproduce all 20.
__tracker = FigureTracker(topic="SignalObjExamples", output_root=OUTPUT_ROOT, expected_count=20)


def _clean(title=None, figsize=(10.0, 6.0)):
    """Strip the FigureTracker placeholder axes and set a clean suptitle.

    FigureTracker.new_figure() seeds every figure with a hidden 1x1 axes that
    carries the ``Topic :: Figure NNN`` title (and any ``matlab_line``
    annotation text).  Because that axes spans the whole figure with
    ``axis_off`` set, any subsequent ``plt.subplot(...)`` or ``plt.gca()``
    overlays on top of it — and the annotation text/title bleed into the
    saved PNG with no ticks or frame.  Round-3 fix: identify the placeholder
    by ``axison == False`` (it is the only axes with axis_off explicitly set)
    rather than by ``has_data()``, since the matlab_line annotation makes the
    placeholder look "non-empty" to the previous heuristic.
    """
    fig = plt.gcf()
    fig.set_size_inches(*figsize)
    tag = None
    for ax in list(fig.axes):
        # Placeholder axes from FigureTracker.new_figure() always has axison
        # turned off and no data lines / images / collections of its own.
        is_placeholder = (not ax.axison) and (
            not ax.lines and not ax.images and not ax.collections and not ax.patches
        )
        if not is_placeholder:
            continue
        if tag is None:
            tag = ax.get_title()
        fig.delaxes(ax)
    if title is None and tag:
        title = tag
    if title:
        fig.suptitle(title, fontsize=11)


def _layout(figsize=(10.0, 6.0), top=0.88, hspace=0.45, wspace=None):
    """Apply consistent figsize + spacing to avoid suptitle/title overlap."""
    fig = plt.gcf()
    fig.set_size_inches(*figsize)
    kwargs = dict(top=top, hspace=hspace)
    if wspace is not None:
        kwargs["wspace"] = wspace
    plt.subplots_adjust(**kwargs)


sampleRate = 100
t = np.arange(0, 10 + 1 / sampleRate, 1 / sampleRate)
freq = 2
v1 = np.sin(2 * np.pi * freq * t)
v2 = np.sin(v1 ** 2)
v = np.vstack([v1, v2])


In [ ]:
# Example 1: Defining and Plotting Signals
s = SignalObj(t, v, "Voltage", "time", "s", "V", ["v1", "v2"])
s1 = SignalObj(t, v1, "Voltage", "time", "s", "V", ["v1"])
__tracker.new_figure("Example 1: define + plot s and s1")
_clean("Example 1: s and s1")
plt.subplot(2, 1, 1); s.plot(); plt.title("s (v1, v2)"); plt.legend(["v1", "v2"])
plt.subplot(2, 1, 2); s1.plot(); plt.title("s1 (v1)"); plt.legend(["v1"])
_layout(top=0.9, hspace=0.5)

# Masking: show only v1, then only v2
__tracker.new_figure("Example 1: masked views of s")
_clean("Example 1: masked s")
plt.subplot(2, 1, 1); s.setMask(["v1"]); s.plot(); s.resetMask(); plt.title("mask -> v1")
plt.subplot(2, 1, 2); s.setMask(["v2"]); s.plot(); s.resetMask(); plt.title("mask -> v2")
_layout(top=0.9, hspace=0.5)

# Repeated dataLabels: two realizations of v1
sr3 = SignalObj(t, np.vstack([v1, v1, v2]), "Voltage", "time", "s", "V", ["v1", "v1", "v2"])
__tracker.new_figure("Example 1: getSubSignal({v1}) -> both realizations")
_clean("Example 1: getSubSignal({'v1'})", figsize=(10.0, 4.5))
ax = plt.gca()
sr3.getSubSignal(["v1"]).plot(handle=ax)
ax.set_title("getSubSignal(v1) - both realizations")
ax.set_xlabel("time [s]")
ax.set_ylabel("Voltage [V]")
ax.legend(["v1", "v1"])
ax.tick_params(left=True, bottom=True, labelleft=True, labelbottom=True)
plt.subplots_adjust(top=0.86, left=0.1, right=0.96, bottom=0.14)


In [ ]:
# Example 2: Changing Signal Properties
s = SignalObj(t, v, "Voltage", "time", "s", "V", ["v1", "v2"])
__tracker.new_figure("Example 2: change x label/units")
_clean("Example 2: change x label/units")
plt.subplot(2, 1, 1); s.plot(); plt.title("original")
plt.subplot(2, 1, 2)
s.setXlabel("distance"); s.setXUnits("cm"); s.plot()
plt.title("xlabel=distance, xunits=cm")
_layout(top=0.9, hspace=0.5)

s = SignalObj(t, v, "Voltage", "time", "s", "V", ["v1", "v2"])
__tracker.new_figure("Example 2: relabel + time window")
_clean("Example 2: relabel + time window")
plt.subplot(2, 1, 1)
s.setDataLabels(["r1", "r2"]); s.setYLabel("Temperature"); s.setYUnits("C"); s.plot()
plt.title("relabelled, ylabel=Temperature, yunits=C"); plt.legend(["r1", "r2"])
plt.subplot(2, 1, 2)
s.setMaxTime(14); s.setMinTime(-2); s.plot()
plt.title("setMaxTime(14), setMinTime(-2)")
_layout(top=0.9, hspace=0.5)

# Per-component styled plots (MATLAB Fig 6):
# subplot(2,1,1) s.plot('v1', {{'k'}})  -> v1 black solid only
# subplot(2,1,2) s.plot('all', {{'k'}, {'-.g'}}) -> v1 black solid + v2 green dash-dot
s = SignalObj(t, v, "Voltage", "time", "s", "V", ["v1", "v2"])
__tracker.new_figure("Example 2: per-component styled plots")
_clean("Example 2: styled plots ('k' / '-.g')")
plt.subplot(2, 1, 1); s.plot("v1", plotPropsIn=["k"]); plt.title("plot v1 with 'k'"); plt.legend(["v1"])
plt.subplot(2, 1, 2); s.plot(plotPropsIn=["k", "-.g"]); plt.title("plot all with 'k' and '-.g'"); plt.legend(["v1", "v2"])
_layout(top=0.9, hspace=0.5)

# Subsets with and without per-component styles (MATLAB Fig 7):
# subplot(2,1,1) s.plot({'v1','v2'})  -> default colors
# subplot(2,1,2) s.plot({'v1','v2'}, {{'k'}, {'-.g'}})  -> styled
s = SignalObj(t, v, "Voltage", "time", "s", "V", ["v1", "v2"])
__tracker.new_figure("Example 2: subset plot + styled subset")
_clean("Example 2: subset plot + styled subset")
plt.subplot(2, 1, 1); s.plot(["v1", "v2"]); plt.title("plot v1 and v2 (defaults)"); plt.legend(["v1", "v2"])
plt.subplot(2, 1, 2); s.plot(["v1", "v2"], plotPropsIn=["k", "-.g"]); plt.title("plot v1, v2 with 'k' and '-.g'"); plt.legend(["v1", "v2"])
_layout(top=0.9, hspace=0.5)


In [ ]:
# Example 3: Resampling and Windowing
s = SignalObj(t, v, "Voltage", "time", "s", "V", ["v1", "v2"])
s1 = s.resample(0.1 * sampleRate)
__tracker.new_figure("Example 3: resample to 0.1*sampleRate")
_clean("Example 3: resample")
plt.subplot(2, 1, 1); s.plot(); plt.title("original (100 Hz)"); plt.legend(["v1", "v2"])
plt.subplot(2, 1, 2); s1.plot(); plt.title("resampled (10 Hz)"); plt.legend(["v1", "v2"])
_layout(top=0.9, hspace=0.5)

__tracker.new_figure("Example 3: getSigInTimeWindow(-2, 3)")
_clean("Example 3: getSigInTimeWindow(-2, 3)")
plt.subplot(2, 1, 1); s.getSigInTimeWindow(-2, 3).plot(); plt.title("window [-2, 3]"); plt.legend(["v1", "v2"])
plt.subplot(2, 1, 2); s.plot(); plt.title("full"); plt.legend(["v1", "v2"])
_layout(top=0.9, hspace=0.5)


In [ ]:
# Example 4: SignalObj Mathematical Operations
s = SignalObj(t, v, "Voltage", "time", "s", "V", ["v1", "v2"])
s5 = SignalObj(t, v - v.mean(axis=1, keepdims=True), "Voltage", "time", "s", "V", ["v1", "v2"])

__tracker.new_figure("Example 4: zero-mean (s - mean(s))")
_clean("Example 4: s - mean(s)", figsize=(10.0, 4.5))
ax = plt.gca()
s5.plot(handle=ax)
ax.set_title("s - mean(s)")
ax.set_xlabel("time [s]"); ax.set_ylabel("Voltage [V]")
ax.legend(["v1-mu(v1)", "v2-mu(v2)"])
ax.set_xticks(np.arange(0, 11, 1))
ax.tick_params(left=True, bottom=True, labelleft=True, labelbottom=True)
plt.subplots_adjust(top=0.86, left=0.1, right=0.96, bottom=0.14)

# mean across dimensions -> single averaged signal
sacross = SignalObj(t, v.mean(axis=0), "Voltage", "time", "s", "V", ["mean"])
__tracker.new_figure("Example 4: mean across dimensions")
_clean("Example 4: mean across dimensions", figsize=(10.0, 4.5))
ax = plt.gca()
sacross.plot(handle=ax)
ax.set_title("mean across dimensions")
ax.set_xlabel("time [s]"); ax.set_ylabel("mu(Voltage) [V]")
ax.legend(["mean"])
ax.tick_params(left=True, bottom=True, labelleft=True, labelbottom=True)
plt.subplots_adjust(top=0.86, left=0.1, right=0.96, bottom=0.14)

s4 = 2 * s + s5
__tracker.new_figure("Example 4: 2*s + s5")
_clean("Example 4: 2*s + (s - mean(s))", figsize=(10.0, 4.5))
ax = plt.gca()
s4.plot(handle=ax)
ax.set_title("2*s + (s - mean(s))")
ax.set_xlabel("time [s]"); ax.set_ylabel("Voltage [V]")
ax.legend(["v1+v1-mu(v1)", "v2+v2-mu(v2)"])
ax.tick_params(left=True, bottom=True, labelleft=True, labelbottom=True)
plt.subplots_adjust(top=0.86, left=0.1, right=0.96, bottom=0.14)

__tracker.new_figure("Example 4: integral / derivative / round-trip")
_clean("Example 4: integral / derivative / round-trip", figsize=(10.0, 8.0))
plt.subplot(3, 1, 1); s.integral().plot(); plt.title("integral")
plt.subplot(3, 1, 2); s.derivative().plot(); plt.title("derivative")
plt.subplot(3, 1, 3); (s.integral().derivative() - s).plot(); plt.title("integral.derivative - s (~0)")
_layout(figsize=(10.0, 8.0), top=0.92, hspace=0.7)


In [ ]:
# Example 5: Spectra
s = SignalObj(t, v, "Voltage", "time", "s", "V", ["v1", "v2"])

# Multi-taper spectrum: MATLAB renders one subplot per signal dimension with
# +/-95% confidence bands (chi-squared CI from MTMspectrum).
f, Smt, Sci = s.MTMspectrum()
__tracker.new_figure("Example 5: multitaper spectrum (per-signal + 95% CI)")
_clean("Example 5: MTM spectrum (per-signal + 95% CI)", figsize=(11.0, 4.5))
labels = ["v1", "v2"]
for i, lbl in enumerate(labels):
    ax = plt.subplot(1, 2, i + 1)
    psd_db = 10 * np.log10(np.abs(Smt[:, i]) + 1e-12)
    lo_db = 10 * np.log10(np.abs(Sci[:, 2 * i]) + 1e-12)
    hi_db = 10 * np.log10(np.abs(Sci[:, 2 * i + 1]) + 1e-12)
    ax.plot(f, psd_db, "b", label=lbl)
    ax.plot(f, hi_db, "y--", label="+95% Conf. Int.")
    ax.plot(f, lo_db, "k--", label="-95% Conf. Int.")
    ax.set_xlabel("Frequency (Hz)")
    ax.set_ylabel("Power (dB/Hz)")
    ax.set_title(f"MTM spectrum ({lbl})")
    ax.set_xlim(0, 50)
    ax.tick_params(left=True, bottom=True, labelleft=True, labelbottom=True)
    ax.legend(loc="upper right", fontsize=8)
plt.subplots_adjust(top=0.84, bottom=0.14, left=0.08, right=0.97, wspace=0.3)

# Periodogram: same per-signal subplot layout, no CI bands.
fp, Pp = s.periodogram()
__tracker.new_figure("Example 5: periodogram (per-signal)")
_clean("Example 5: periodogram (per-signal)", figsize=(11.0, 4.5))
for i, lbl in enumerate(labels):
    ax = plt.subplot(1, 2, i + 1)
    ax.plot(fp, 10 * np.log10(np.abs(Pp[:, i]) + 1e-12), "b", label=lbl)
    ax.set_xlabel("Frequency (Hz)")
    ax.set_ylabel("Power (dB/Hz)")
    ax.set_title(f"periodogram ({lbl})")
    ax.set_xlim(0, 50)
    ax.tick_params(left=True, bottom=True, labelleft=True, labelbottom=True)
    ax.legend(loc="upper right", fontsize=8)
plt.subplots_adjust(top=0.84, bottom=0.14, left=0.08, right=0.97, wspace=0.3)


In [ ]:
# Example 6: Signal Variability
sampleRate6 = 5000
t6 = np.arange(0, 1 + 1 / sampleRate6, 1 / sampleRate6)
v1b = np.sin(2 * np.pi * 2 * t6)
v2b = np.sin(v1b ** 2)
noise = 0.1 * np.random.randn(len(t6), 6)
data = np.vstack([v1b, v2b, v2b, v1b, v2b, v1b]).T + noise
sv = SignalObj(t6, data.T, "Voltage", "time", "s", "V", ["v1", "v2", "v2", "v1", "v1", "v2"])

# MATLAB Fig 16: all signals on top, single all-variability patch on bottom.
__tracker.new_figure("Example 6: signals + all-variability")
_clean("Example 6: signals + plotAllVariability")
ax1 = plt.subplot(2, 1, 1); sv.plot(handle=ax1); ax1.set_title("noisy signals")
ax1.set_xlabel("time [s]"); ax1.set_ylabel("Voltage [V]")
ax2 = plt.subplot(2, 1, 2); sv.plotAllVariability(ax=ax2); ax2.set_title("plotAllVariability")
ax2.set_xlabel("time [s]"); ax2.set_ylabel("Voltage [V]")
_layout(top=0.9, hspace=0.5)

# MATLAB Fig 17 -- plotVariability renders ONE figure per unique label.
# Pass an explicit Axes handle (and call _clean BEFORE plotting) so the
# placeholder is removed before plotAllVariability calls plt.gca() — otherwise
# the v2 plot lands on the v1 axes and the two figures merge into one PNG.
sv_v1 = sv.getSubSignal(["v1"])
__tracker.new_figure("Example 6: plotVariability (v1)")
_clean("Example 6: plotVariability (v1)", figsize=(10.0, 4.5))
ax = plt.gca()
sv_v1.plotAllVariability(faceColor="tab:blue", ax=ax)
ax.set_title("plotVariability v1")
ax.set_xlabel("time [s]"); ax.set_ylabel("Voltage [V]")
ax.tick_params(left=True, bottom=True, labelleft=True, labelbottom=True)
plt.subplots_adjust(top=0.86, left=0.1, right=0.96, bottom=0.14)

sv_v2 = sv.getSubSignal(["v2"])
__tracker.new_figure("Example 6: plotVariability (v2)")
_clean("Example 6: plotVariability (v2)", figsize=(10.0, 4.5))
ax = plt.gca()
sv_v2.plotAllVariability(faceColor="tab:orange", ax=ax)
ax.set_title("plotVariability v2")
ax.set_xlabel("time [s]"); ax.set_ylabel("Voltage [V]")
ax.tick_params(left=True, bottom=True, labelleft=True, labelbottom=True)
plt.subplots_adjust(top=0.86, left=0.1, right=0.96, bottom=0.14)

# MATLAB Fig 19 (Fig 18 is a publish() duplicate of Fig 16 - skipped here).
__tracker.new_figure("Example 6: customized variability bands")
_clean("Example 6: customized variability bands", figsize=(10.0, 8.0))
ax1 = plt.subplot(3, 1, 1); sv.plotAllVariability("b", ax=ax1); ax1.set_title("b")
ax1.set_xlabel("time [s]"); ax1.set_ylabel("Voltage [V]")
ax2 = plt.subplot(3, 1, 2); sv.plotAllVariability("g", 2, ax=ax2); ax2.set_title("g, lw=2")
ax2.set_xlabel("time [s]"); ax2.set_ylabel("Voltage [V]")
ax3 = plt.subplot(3, 1, 3); sv.plotAllVariability("c", 3, 2, 1, ax=ax3); ax3.set_title("c, lw=3, CI 2/1")
ax3.set_xlabel("time [s]"); ax3.set_ylabel("Voltage [V]")
_layout(figsize=(10.0, 8.0), top=0.92, hspace=0.7)

# MATLAB Fig 20: side-by-side plotVariability(v1, v2) summary -- the 20th
# helpfile PNG renders the per-label variability bands together so the
# two-realization-group widths can be compared directly. MATLAB itself
# achieves this by repeating ``s.plotVariability`` after the customized
# figure; we render both labels on a single 1x2 subplot grid here.
__tracker.new_figure("Example 6: plotVariability(v1, v2) side-by-side")
_clean("Example 6: plotVariability v1 and v2 (side-by-side)", figsize=(11.0, 4.5))
ax_a = plt.subplot(1, 2, 1)
sv_v1.plotAllVariability(faceColor="tab:blue", ax=ax_a)
ax_a.set_title("plotVariability v1")
ax_a.set_xlabel("time [s]"); ax_a.set_ylabel("Voltage [V]")
ax_a.tick_params(left=True, bottom=True, labelleft=True, labelbottom=True)
ax_b = plt.subplot(1, 2, 2)
sv_v2.plotAllVariability(faceColor="tab:orange", ax=ax_b)
ax_b.set_title("plotVariability v2")
ax_b.set_xlabel("time [s]"); ax_b.set_ylabel("Voltage [V]")
ax_b.tick_params(left=True, bottom=True, labelleft=True, labelbottom=True)
plt.subplots_adjust(top=0.84, bottom=0.14, left=0.07, right=0.97, wspace=0.25)

__tracker.finalize()
